# Lecture 3 — Loading and Exporting CSV and JSON Data with NumPy and Pandas

This notebook demonstrates standard approaches for loading and exporting tabular data in Python:
1. **NumPy** — suited for homogeneous numerical arrays
2. **pandas** — suited for heterogeneous, labelled tabular data

We cover CSV files (loading and exporting) as well as JSON files (loading and exporting).

We use a small product sales dataset (`sales_data.csv`) throughout. Note, as it is stored in a different folder than where this Jupyter Notebook is, we must specify the path. In this case it is simply done by adding `data/` in front of the file name so that the file is reached in the folder `data`.

**Credits:**

Created by Sippo Rossi for the course Python Programming for Business Intelligence at Hanken.

## 1. Loading CSV data with NumPy

In [1]:
import numpy as np

### 1.1 `np.loadtxt()` — Numeric Columns Only

`loadtxt` works best when every value is numeric. We skip the header row and select only the numeric columns (indices 2, 3, 4).

In [2]:
numeric_data = np.loadtxt(
    "data/sales_data.csv",
    delimiter=",",
    skiprows=1,          # skip header
    usecols=(2, 3, 4)    # units_sold, unit_price, revenue
)

print("Shape:", numeric_data.shape)
print("Dtype:", numeric_data.dtype)
print()
print(numeric_data[:5])

Shape: (10, 3)
Dtype: float64

[[1.200000e+02 8.999900e+02 1.079988e+05]
 [4.500000e+02 4.999000e+01 2.249550e+04]
 [2.000000e+02 2.499900e+02 4.999800e+04]
 [1.800000e+02 3.499900e+02 6.299820e+04]
 [1.500000e+03 3.490000e+00 5.235000e+03]]


### 1.2 `np.genfromtxt()` — Handling Mixed Types

`genfromtxt` can handle strings and missing values via structured arrays. Setting `dtype=None` lets NumPy infer types per column.

In [3]:
structured_data = np.genfromtxt(
    "data/sales_data.csv",
    delimiter=",",
    names=True,          # use first row as field names
    dtype=None,          # auto-detect types per column
    encoding="utf-8"
)

print("Field names:", structured_data.dtype.names)
print("Number of records:", structured_data.shape[0])
print()

# Access columns by name
print("Products:", structured_data["product"][:3])
print("Revenue: ", structured_data["revenue"][:3])

Field names: ('product', 'category', 'units_sold', 'unit_price', 'revenue')
Number of records: 10

Products: ['Laptop' 'Keyboard' 'Desk Chair']
Revenue:  [107998.8  22495.5  49998. ]


## 2. Loading CSV data with pandas

In [4]:
import pandas as pd

### 2.1 `pd.read_csv()` The standard approach

`read_csv` is the most common entry point for tabular data in Python. It infers headers, data types, and handles many edge cases automatically.

In [5]:
df = pd.read_csv("data/sales_data.csv")

print("Shape:", df.shape)
print()
print(df.dtypes)

Shape: (10, 5)

product        object
category       object
units_sold      int64
unit_price    float64
revenue       float64
dtype: object


In [6]:
df.head()

,product,category,units_sold,unit_price,revenue
0,Laptop,Electronics,120,899.99,107998.8
1,Keyboard,Electronics,450,49.99,22495.5
2,Desk Chair,Furniture,200,249.99,49998.0
3,Monitor,Electronics,180,349.99,62998.2
4,Notebook,Stationery,1500,3.49,5235.0


### 2.2 Optional `read_csv` parameters

A few examples of parameters that can be used:

In [7]:
# Select specific columns and set an index

df_subset = pd.read_csv(
    "data/sales_data.csv",
    usecols=["product", "category", "revenue"],
    index_col="product"
)

df_subset.head()

,category,revenue
product,,
Laptop,Electronics,107998.8
Keyboard,Electronics,22495.5
Desk Chair,Furniture,49998.0
Monitor,Electronics,62998.2
Notebook,Stationery,5235.0


### 2.3 Exporting CSV files

Once data has been processed, it is common to save the result as a new CSV file. Both NumPy and pandas provide straightforward export functions.

In [8]:
# Export a NumPy array to CSV
np.savetxt(
    "data/numeric_export.csv",
    numeric_data,
    delimiter=",",
    header="units_sold,unit_price,revenue",
    comments=""       # avoid the default '# ' prefix on the header
)

print("NumPy array exported to data/numeric_export.csv")

NumPy array exported to data/numeric_export.csv


In [9]:
# Export a pandas DataFrame to CSV
df.to_csv("data/sales_export.csv", index=False)

print("DataFrame exported to data/sales_export.csv")

DataFrame exported to data/sales_export.csv


Setting `index=False` prevents pandas from writing the row index as an extra column, which is usually the desired behaviour when creating a clean export file.

## 3. Loading and Exporting JSON Data

JSON (JavaScript Object Notation) is another common data format, especially for data retrieved from web APIs. Unlike CSV, JSON can naturally represent **nested structures**, for example, a customer record that contains a list of orders.

We use a small customer–orders dataset (`customers.json`) for this section. Python's built-in `json` module handles raw JSON, while pandas can read JSON directly into a DataFrame.

Let us first inspect the raw JSON file to see its structure:

In [10]:
# Display the raw contents of the JSON file
with open("data/customers.json", "r") as f:
    raw_text = f.read()

print(raw_text)

[
  {
    "customer_id": "C001",
    "name": "Anna Lindberg",
    "city": "Helsinki",
    "orders": [
      {
        "order_id": "A101",
        "product": "Laptop",
        "amount": 899.99
      },
      {
        "order_id": "A102",
        "product": "Mouse",
        "amount": 24.5
      }
    ]
  },
  {
    "customer_id": "C002",
    "name": "Erik Holm",
    "city": "Stockholm",
    "orders": [
      {
        "order_id": "A103",
        "product": "Monitor",
        "amount": 349.99
      }
    ]
  },
  {
    "customer_id": "C003",
    "name": "Maria Korhonen",
    "city": "Helsinki",
    "orders": [
      {
        "order_id": "A104",
        "product": "Keyboard",
        "amount": 49.99
      },
      {
        "order_id": "A105",
        "product": "Desk Chair",
        "amount": 249.99
      },
      {
        "order_id": "A106",
        "product": "Webcam",
        "amount": 79.0
      }
    ]
  },
  {
    "customer_id": "C004",
    "name": "Lars Pettersson",
    "city": "

### 3.1 Loading JSON with the `json` module

The standard library `json` module reads a JSON file into native Python objects (dictionaries and lists). This is useful when the data has nested structures that do not map directly to a flat table.

In [12]:
import json

with open("data/customers.json", "r") as f:
    customers = json.load(f)

print("Type:", type(customers))
print("Number of records:", len(customers))
print()
print("First record:")
print(customers[0])

Type: <class 'list'>
Number of records: 5

First record:
{'customer_id': 'C001', 'name': 'Anna Lindberg', 'city': 'Helsinki', 'orders': [{'order_id': 'A101', 'product': 'Laptop', 'amount': 899.99}, {'order_id': 'A102', 'product': 'Mouse', 'amount': 24.5}]}


Notice that each customer record contains a nested list of orders. This kind of hierarchy is straightforward in JSON but impossible to represent directly in a flat CSV file.

### 3.2 Loading JSON with pandas

`pd.read_json()` reads a JSON file directly into a DataFrame. When the JSON contains nested objects, pandas stores them as Python objects inside the cells.

In [13]:
df_customers = pd.read_json("data/customers.json")

print("Shape:", df_customers.shape)
print()
print(df_customers.dtypes)
print()
df_customers.head()

Shape: (5, 4)

customer_id    object
name           object
city           object
orders         object
dtype: object



,customer_id,name,city,orders
0,C001,Anna Lindberg,Helsinki,"[{'order_id': 'A101', 'product': 'Laptop', 'am..."
1,C002,Erik Holm,Stockholm,"[{'order_id': 'A103', 'product': 'Monitor', 'a..."
2,C003,Maria Korhonen,Helsinki,"[{'order_id': 'A104', 'product': 'Keyboard', '..."
3,C004,Lars Pettersson,Gothenburg,"[{'order_id': 'A107', 'product': 'Laptop', 'am..."
4,C005,Johanna Niemi,Turku,"[{'order_id': 'A109', 'product': 'Tablet', 'am..."


The `orders` column contains Python lists of dictionaries. To analyse the individual orders, one option is `pd.json_normalize()`, which flattens nested records into a regular table:

In [14]:
df_orders = pd.json_normalize(
    customers,
    record_path="orders",          # the nested list to flatten
    meta=["customer_id", "name"]    # parent fields to keep
)

df_orders.head()

,order_id,product,amount,customer_id,name
0,A101,Laptop,899.99,C001,Anna Lindberg
1,A102,Mouse,24.50,C001,Anna Lindberg
2,A103,Monitor,349.99,C002,Erik Holm
3,A104,Keyboard,49.99,C003,Maria Korhonen
4,A105,Desk Chair,249.99,C003,Maria Korhonen


### 3.3 Exporting JSON

Exporting follows the same pattern: use the `json` module for raw Python objects, or `DataFrame.to_json()` for pandas.

In [15]:
# Export using the json module
with open("data/customers_export.json", "w") as f:
    json.dump(customers, f, indent=2)

print("Exported via json module to data/customers_export.json")

Exported via json module to data/customers_export.json


In [16]:
# Export the flattened orders DataFrame using pandas
df_orders.to_json("data/orders_export.json", orient="records", indent=2)

print("Exported via pandas to data/orders_export.json")

Exported via pandas to data/orders_export.json


The `orient="records"` parameter tells pandas to write the data as a list of dictionaries (one per row), which is the most common JSON structure for tabular data. The `indent=2` parameter makes the output human-readable.

## 4. When to use NumPy and when to use pandas

| Criterion | NumPy | pandas |
|---|---|---|
| Data type | Homogeneous numeric | Mixed / heterogeneous |
| Column labels | Manual (structured arrays) | Built-in (`DataFrame`) |
| Missing data | Limited (`genfromtxt`) | Native support (`NaN`) |

**Rule of thumb:** use pandas for exploratory data analysis and data wrangling; use NumPy when working with purely numerical arrays or when performance on large-scale computation is critical.